## Overview
This script generates the average stroke for each gesture's left and right controller.

### Output Details
For each code block, a total of 26 files are exported: two for each gesture, representing either freeform or instructional data. These files include data from the first five trials of each participant session. Each row corresponds to data from an individual stroke or trial. A blank row is used to separate different participant sessions. Additionally, there are two columns that represent the left and right controllers.

### Metric Calculation
    * Length (Code block 5)
    * Duration (Code block 6)
    * Speed (Code block 7)
    * Acceleration (Code block 8)
    * Angle (Code block 9)
    * Curvature (Code block 10)

### Folder Directory
To simplify the file exports, making the following folders for the output metric calculations:
    * GestureAcceleration
    * GestureAngle
    * GestureCurvature
    * GestureDuration
    * GestureLength
    * GestureSpeed

### How to Run
Run code blocks in order
* Create folders in the directory
* Import libraries (#1)
* Generate gesture dictionary with file paths to the data (#2)
* Define mutual functions for metric calculations (#3)
* Helper methods to remove certain strokes and extremities from calculations (#4)
* Functions for metric calculation (#5 - #10)
* Functions to iterate through data files and run the calculations (#11)
* Run the methods here (#12)

Read the markdowns to ensure that edits have been made.

In [34]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import BSpline, make_interp_spline
import plotly.io as pio

## Overview #1
Create a dictionary with the gestures as keys and a list of data file path as the values.

['PanDown', 'PanLeft', 'PanRight', 'PanUp', 'RotBackX', 'RotClockwiseY', 'RotCounterY', 'RotForwardX', 'RotLeftZ', 'RotRightZ', 'ZoomIn', 'ZoomOut']

### To Do
* Update cleaned_data_folder_path variable


In [35]:
''' Edit variable here'''
cleaned_data_folder_path = 'C:\\Users\\katie\\Documents\\cpsc\\VRelax\\gestureInterface\\CleanedData'

gesture_dict = {}

# Iterate through CleanedData directory to create a dictionary of gestures
# Each of the 12 gesture has two lists (freeform and instructional) containing paths to CSV data files.
for root, sub_folders, files in os.walk(cleaned_data_folder_path):
    for file in files:
        # Splits file name for gesture and session type identification
        file_split = file.split("_") 
        file_gesture, file_session_type, subject_id = file_split[3], file_split[2], file_split[5]

        # Skip unnecessary files: Box Select or Subject 1
        if file_gesture == 'BoxSelect' or subject_id == '1':
            continue

        # Skip the thank you/non-session files
        if file_session_type not in {'F', 'I'}:
            continue

        # Initialize gesture key in the dictionary if not already present
        if file_gesture not in gesture_dict:
            gesture_dict[file_gesture] = ([], [])  # [freeform_list, instructional_list]

        # Append file path to the appropriate session list
        if file_session_type == 'F':
            gesture_dict[file_gesture][0].append(os.path.join(root, file))
        else:
            gesture_dict[file_gesture][1].append(os.path.join(root, file))
gesture_dict = dict(sorted(gesture_dict.items()))

for gesture, (freeform, instructional) in gesture_dict.items():
    gesture_dict[gesture] = (sorted(freeform), sorted(instructional))

### Overview #2
Functions to remove specific strokes based on stroke removal csv file.

## To Do
* Edit df_for_stroke_removal file path


In [36]:
'''Edit variable here'''
df_for_stroke_removal = pd.read_csv('C:\\Users\\katie\\Documents\\cpsc\\VRelax\\gestureInterface\\MetricCalculations\\strokes_to_remove.csv')


def disect_cell(input):
    """
    Determines which strokes to remove based on cell value in the stroke removal file.

    Parameters
    -----
    input : str
        value in cell. Has the format of [3:5], [2], or [4:] where the number indicates the stroke and ':' indicates a range.

    Returns
    -----
    list_strokes_to_remove : str
        list of strokes to remove
    """

    i, list_strokes_to_remove = 0, []
    while i < len(input):
        if input[i] == '[':
            i += 1
            # Indicates a range of strokes to be removed
            if input[i+1] == ':':
                for j in range(int(input[i]), int(input[i+2])+1):
                    list_strokes_to_remove.append(j)
                i += 1
            # A single stroke is to be removed
            else:
                list_strokes_to_remove.append(int(input[i]))
                i += 1
        else:
            i += 1

    return list_strokes_to_remove


def remove_strokes(df, gesture_key_index, row_index):
    """
    Removes strokes by setting trigger pulls in that trial to 0.

    Parameters
    -----
    df : dataframe
        dataframe for a single sub session data file such as sub 6 session 3.
    gesture_key_index : int
        index of the gesture, which will be used to access the correct column in the stroke removal file.
    row_index : int
        index of the row that indicates which subject session. Will be used to access the correct row in the stroke removal file.
    """

    l_col_index, r_col_index = gesture_key_index * 2 + 1, gesture_key_index * 2 + 2
    value_in_left_cell = df_for_stroke_removal.iloc[row_index, l_col_index]
    value_in_right_cell = df_for_stroke_removal.iloc[row_index, r_col_index]

    # Determine which strokes to remove in that list
    l_strokes_to_remove = disect_cell(value_in_left_cell) if pd.notna(value_in_left_cell) else []
    r_strokes_to_remove = disect_cell(value_in_right_cell) if pd.notna(value_in_right_cell) else []


    # Change original df's trigger_pull_amount to 0 where strokes are to be removed
    for stroke_num in l_strokes_to_remove:
        df.loc[df['gesture_counter_UI'] == stroke_num, 'trigger_pull_amount_left'] = 0
    for stroke_num in r_strokes_to_remove:
        df.loc[df['gesture_counter_UI'] == stroke_num, 'trigger_pull_amount_right'] = 0

### Overview #3
Functions to plot a stroke. Contains three different types of plotting: markers, average (uses a timescale coloring to show change in time), lines

In [37]:
def plot_markers(fig, coordinates, stroke_color):
    x_vals, y_vals, z_vals = zip(*coordinates)
    fig.add_trace(go.Scatter3d(
    x=x_vals,
    y=y_vals,
    z=z_vals,
    mode='markers',
    marker=dict(
        size=2,
        color=stroke_color,  # colorscale
        opacity=0.8
    ),
    showlegend=True)
)
    
def plot_avg(fig, coordinates):
    indices = np.arange(len(coordinates))
    x_vals, y_vals, z_vals = zip(*coordinates)
    
    fig.add_trace(go.Scatter3d(
    x=x_vals,
    y=y_vals,
    z=z_vals,
    mode='markers',
    name='avg stroke',
    marker=dict(
        size=2,
        color=indices,  # Assign indices to color for gradient effect
        colorscale='Viridis',  # You can change to other color scales like 'Jet', 'Plasma', etc.
        opacity=0.8,
        colorbar=dict(
                tickvals=[indices[0], indices[-1]],  # Set custom tick positions
                ticktext=["Start", "End"],  # Label these positions
                len=0.6,
                x=.95,
                y=0.9,
            ),
    ),
    showlegend=True)
)
    
def plot_line(fig, coordinates, stroke_color):
    x_vals, y_vals, z_vals = zip(*coordinates)
    fig.add_trace(go.Scatter3d(
    x=x_vals,
    y=y_vals,
    z=z_vals,
    mode='lines',
    marker=dict(
        size=2,
        color=stroke_color,  # colorscale
        opacity=0.8,
    ),
    showlegend=True)
)


### Overview #4
Functions to remove rows where controller trigger is not pulled and where the trial number does not match

In [38]:
def drop_df(df, trial_num):
    df.drop(df[(df['trigger_pull_amount_left'] == 0) & (df['trigger_pull_amount_right'] == 0)].index, inplace=True)
    df.drop(df[(df['gesture_counter_UI']) != trial_num].index, inplace=True)
    df.reset_index(drop=True, inplace=True)

# Removes rows where none of the triggers are pulled
# Removes rows wthat are not equal to the trial
# Remove rows where there may be na
def get_cleaned_df(df, trial_num, hand):
    trigger_col = f"trigger_pull_amount_{'left' if hand == 'l' else 'right'}"
    condition = (df[trigger_col] != 0) & (df['gesture_counter_UI'] == trial_num)
    condition &= df[hand + "_controller_translation_x"].notna()
    filtered_df = df[condition].reset_index(drop=True)

    return filtered_df

def get_data_points(df, hand):
    x = np.array(df[hand + '_controller_translation_x'])
    y = np.array(df[hand + '_controller_translation_y'])
    z = np.array(df[hand + '_controller_translation_z'])

    return list(zip(x,y,z))

def get_head_translation(df):
    x = np.array(df['head_translation_x'])
    y = np.array(df['head_translation_y'])
    z = np.array(df['head_translation_z'])

    return list(zip(x,y,z))


### Overview #5
Functions to apply egocentric coordinates

In [39]:
selected_columns = ['r_controller_translation_x', 'r_controller_translation_y', 'r_controller_translation_z',
                    'r_controller_rotation_x', 'r_controller_rotation_y', 'r_controller_rotation_z','r_controller_rotation_w',
                    'l_controller_translation_x', 'l_controller_translation_y', 'l_controller_translation_z',
                    'l_controller_rotation_x', 'l_controller_rotation_y', 'l_controller_rotation_z','l_controller_rotation_w',                    
                    'head_translation_x', 'head_translation_y','head_translation_z',
                    'head_rotation_x','head_rotation_y', 'head_rotation_z', 'head_rotation_w']


## 3.1 L-handed -> R-handed (only invert translation_z for head + L/R hands)
def left_to_right_handed(data_sample, selected_columns):
    r_handed_data_sample = [] # r-handed data of the chosen sample; 21 lists
    for idx, col in enumerate(selected_columns):
        if 'translation_z' in col:
            inv_lst = [-float(val) for val in data_sample[idx]]
            
            r_handed_data_sample.append(inv_lst)
        else: # other columns not needing inverted
            r_handed_data_sample.append(data_sample[idx])
    return r_handed_data_sample



## 3.2 Data processing: Rotation 4-Quaternion to 3-direction for head + L/R hands
directional_data_names = ['r_translation_x', 'r_translation_y', 'r_translation_z', 'r_direction_x', 'r_direction_y', 'r_direction_z',
                         'l_translation_x', 'l_translation_y', 'l_translation_z', 'l_direction_x', 'l_direction_y', 'l_direction_z',
                         'head_translation_x', 'head_translation_y', 'head_translation_z', 'head_direction_x', 'head_direction_y', 'head_direction_z']

# function: Convert a quaternion into a 3D rotation matrix
def quaternion_rotation_matrix(Q):
    # Extract values from Q
    qx = float(Q[0])
    qy = float(Q[1])
    qz = float(Q[2])
    qw = float(Q[3])

    # First row of the rotation matrix
    r00 = 1.0 - 2.0 * (qy * qy + qz * qz)
    r01 = 2.0 * (qx * qy - qw * qz)
    r02 = 2.0 * (qx * qz + qw * qy)

    # Second row of the rotation matrix
    r10 = 2.0 * (qx * qy + qw * qz)
    r11 = 1.0 - 2.0 * (qx * qx + qz * qz)
    r12 = 2.0 * (qy * qz - qw * qx)

    # Third row of the rotation matrix
    r20 = 2.0 * (qx * qz - qw * qy)
    r21 = 2.0 * (qy * qz + qw * qx)
    r22 = 1.0 - 2.0 * (qx * qx + qy * qy)

    # 3x3 rotation matrix
    rot_matrix = np.array([[r00, r01, r02],
                           [r10, r11, r12],
                           [r20, r21, r22]])
    return rot_matrix

# function: Convert rotation data represented as quaterinons into a directions (3D vector)
#            by rotating a forward vector (0, 0, 1) using the given quaternion
def convertQuaternions2Directions(rotation_x_list, rotation_y_list,rotation_z_list,rotation_w_list):
    direction_x_list = []
    direction_y_list = []
    direction_z_list = []
    forward_vec = np.array([0, 0, 1])
    for i in range(len(rotation_x_list)):
        quaternion = [rotation_x_list[i],rotation_y_list[i],rotation_z_list[i],rotation_w_list[i]]
        rot_matrix = quaternion_rotation_matrix(quaternion)
        dir_vec = rot_matrix.dot(forward_vec)
        direction_x_list.append(dir_vec[0])
        direction_y_list.append(dir_vec[1])
        direction_z_list.append(dir_vec[2])
    return direction_x_list, direction_y_list, direction_z_list

# Apply to rotations of head and L/R
def quaternion_to_direction(r_handed_data_sample, selected_columns):
    directional_data_sample = []
    items = [[[0, 2], [3, 6]], [[7, 9], [10, 13]], [[14, 16], [17, 20]]] # index range for R/L/Head:[R:[trans,rot],L:[trans,rot],Head:[trans,rot]]
    # traverse R -> L -> Head in order
    for item in items: 
        # translation data
        for idx in range(item[0][0], item[0][1] + 1):
            directional_data_sample.append(r_handed_data_sample[idx])
        # rotation data
        quaternion_lists = []
        direction_x_list = []
        direction_y_list = []
        direction_z_list = []
        for idx in range(item[1][0], item[1][1] + 1):
            quaternion_lists.append(r_handed_data_sample[idx])
        if (len(quaternion_lists) == 4):
            direction_x_list, direction_y_list, direction_z_list = convertQuaternions2Directions(
                quaternion_lists[0], quaternion_lists[1], quaternion_lists[2], quaternion_lists[3])
            directional_data_sample.append(direction_x_list)
            directional_data_sample.append(direction_y_list)
            directional_data_sample.append(direction_z_list)
    return directional_data_sample





## 3.3 World -> Egocentric coordinates (head + L/R hands -> L/R hands) 
#   and reformat the sample as [[...],[...],...[...]], a list of 12 signals (lists)
egocentric_data_names = ['r_translation_u', 'r_translation_v', 'r_translation_w', 'r_direction_u', 'r_direction_v', 'r_direction_w',
                         'l_translation_u', 'l_translation_v', 'l_translation_w', 'l_direction_u', 'l_direction_v', 'l_direction_w']

# function construct a head space coordinate system consisting of three basis vectors, u, v, w for EACH trial
def buildHeadSpaceCoordVectors(head_direction_x_list, head_direction_y_list, head_direction_z_list):
    # average head direction vectors across all frames in the trial
    avg_head_direction_x = sum(head_direction_x_list)/len(head_direction_x_list)
    avg_head_direction_y = sum(head_direction_y_list)/len(head_direction_y_list)
    avg_head_direction_z = sum(head_direction_z_list)/len(head_direction_z_list)
    
    # calculate head space coordinate vectors, u, v, w
    head_space_w = -1.0 * np.array([avg_head_direction_x, avg_head_direction_y, avg_head_direction_z]) # w is opposite of the head direction
    head_space_v = np.array([0, 1, 0])
    head_space_u = np.cross(head_space_v, head_space_w)
    head_space_v = np.cross(head_space_w, head_space_u)

    return head_space_u, head_space_v, head_space_w

# function: convert hand data from world to head
def convert_hand_world_to_head(avg_head_translation, rot_matrix, hand_translation_x_list,  hand_translation_y_list,  hand_translation_z_list,
                                hand_direction_x_list,  hand_direction_y_list,  hand_direction_z_list):
    hand_translation_u_list = []
    hand_translation_v_list = []
    hand_translation_w_list = []
    hand_direction_u_list = []
    hand_direction_v_list = []
    hand_direction_w_list = []

    # convert hand data world -> space, frame by frame
    for idx in range(len(hand_translation_x_list)):
        # Translation data: (Already correct)
        hand_translation_u = float(hand_translation_x_list[idx]) - float(avg_head_translation[0])
        hand_translation_v = float(hand_translation_y_list[idx]) - float(avg_head_translation[1])
        hand_translation_w = float(hand_translation_z_list[idx]) - float(avg_head_translation[2])
        hand_translation = rot_matrix.dot(np.array([hand_translation_u, hand_translation_v, hand_translation_w]))
        hand_translation_u_list.append(hand_translation[0])
        hand_translation_v_list.append(hand_translation[1])
        hand_translation_w_list.append(hand_translation[2])

        # Direction data: (Change this to point toward the origin)
        # Compute the direction from the hand position to the origin
        direction_to_head = -1 * np.array([hand_translation_u, hand_translation_v, hand_translation_w])  # Direction to the origin
        hand_direction_u_list.append(direction_to_head[0])
        hand_direction_v_list.append(direction_to_head[1])
        hand_direction_w_list.append(direction_to_head[2])

    return hand_translation_u_list, hand_translation_v_list, hand_translation_w_list, hand_direction_u_list, hand_direction_v_list, hand_direction_w_list

def world_to_headspace(directional_data_sample, directional_data_names):
    egocentric_data_sample = []
    # obtain head translation data in world space
    # converts to float bc for some reason, some of the data is are strings
    head_translation_x_list = [float(i) for i in directional_data_sample[12]]
    head_translation_y_list = [float(i) for i in directional_data_sample[13]]
    head_translation_z_list = [float(i) for i in directional_data_sample[14]]

    # average head translation data -> averaged head position in world
    avg_head_translation = []
    avg_head_translation.append(sum(head_translation_x_list)/len(head_translation_x_list))
    avg_head_translation.append(sum(head_translation_y_list)/len(head_translation_y_list))
    avg_head_translation.append(sum(head_translation_z_list)/len(head_translation_z_list))

    # obtain head direction data in world space
    head_direction_x_list = directional_data_sample[15]
    head_direction_y_list = directional_data_sample[16]
    head_direction_z_list = directional_data_sample[17]
    
    # build coordiante vectors of the head space: u, v, w
    head_space_u, head_space_v, head_space_w = buildHeadSpaceCoordVectors(head_direction_x_list=head_direction_x_list, 
                                                                          head_direction_y_list=head_direction_y_list,
                                                                          head_direction_z_list=head_direction_z_list)

    # construct rotation matrix transforming vectors from world to head space
    rot_matrix = np.array([head_space_u.tolist(), # u
                           head_space_v.tolist(), # v
                           head_space_w.tolist()]) # w
    
    # convert RIGHT hand data from world to head space
    r_translation_u_list, r_translation_v_list, r_translation_w_list, \
       r_direction_u_list,r_direction_v_list, r_direction_w_list \
            = convert_hand_world_to_head(avg_head_translation, rot_matrix, directional_data_sample[0], directional_data_sample[1],directional_data_sample[2],\
                                         directional_data_sample[3], directional_data_sample[4],directional_data_sample[5])

    # convert LEFT hand data from world to head space
    l_translation_u_list, l_translation_v_list, l_translation_w_list, \
        l_direction_u_list, l_direction_v_list, l_direction_w_list \
            = convert_hand_world_to_head(avg_head_translation, rot_matrix, directional_data_sample[6], directional_data_sample[7],directional_data_sample[8],\
                                         directional_data_sample[9], directional_data_sample[10],directional_data_sample[11])
    
    # add the converted data
    egocentric_data_sample.append(r_translation_u_list) # right hand
    egocentric_data_sample.append(r_translation_v_list)
    egocentric_data_sample.append(r_translation_w_list)
    egocentric_data_sample.append(r_direction_u_list)
    egocentric_data_sample.append(r_direction_v_list)
    egocentric_data_sample.append(r_direction_w_list)
    egocentric_data_sample.append(l_translation_u_list) # left hand
    egocentric_data_sample.append(l_translation_v_list)
    egocentric_data_sample.append(l_translation_w_list)
    egocentric_data_sample.append(l_direction_u_list)
    egocentric_data_sample.append(l_direction_v_list)
    egocentric_data_sample.append(l_direction_w_list)

    return egocentric_data_sample

### Overview #6
Functions to get average stroke and average direction

In [40]:
def get_avg_stroke(list_of_strokes):
    avg_stroke = []
    # Iterate over each index in the stroke length
    for i in range(len(list_of_strokes[0])):
        # Gather the coordinates at the ith index from each stroke
        coordinates_at_index = [stroke[i] for stroke in list_of_strokes]
        
        # Calculate the mean for x, y, and z separately
        avg_x = np.mean([coord[0] for coord in coordinates_at_index])
        avg_y = np.mean([coord[1] for coord in coordinates_at_index])
        avg_z = np.mean([coord[2] for coord in coordinates_at_index])
        
        # Append the average coordinate as a tuple
        avg_stroke.append((avg_x, avg_y, avg_z))

    return avg_stroke

def get_avg_direction(directions_list):
    num_coords = len(directions_list)

    sum_x = sum(float(direction[0]) for direction in directions_list)
    sum_y = sum(float(direction[1]) for direction in directions_list)
    sum_z = sum(float(direction[2]) for direction in directions_list)

    avg_x = sum_x / num_coords
    avg_y = sum_y / num_coords
    avg_z = sum_z / num_coords

    return (avg_x, avg_y, avg_z)

### Overview #7
Drawing left and right controller with headset position and avg direction for each subject session in a single gesture.

### To Do
* Edit variables 'gesture' to any key in the dictionary and 'gesture_index' to its corresponding index

In [41]:
gesture = "RotRightZ"
gesture_index = 9

fig_1 = go.Figure()

for file_num, file_path in enumerate(gesture_dict[gesture][0]):
    df = pd.read_csv(file_path)
    remove_strokes(df, gesture_index, file_num)
    filtered_df_L = get_cleaned_df(df, 1, 'l')
    filtered_df_R = get_cleaned_df(df, 1, 'r')

    if (len(filtered_df_L) != 0):
        data_sample = [] # original data of the chosen sample; 21 lists
        for col in selected_columns:
            data_sample.append(filtered_df_L[col].tolist())

        # 3.1
        #r_handed_data_sample = left_to_right_handed(data_sample=data_sample, selected_columns=selected_columns)
        # 3.2
        directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
        # 3.3
        #egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)

        pts_from_3_2 = (list(zip(directional_data_sample[6], directional_data_sample[7], directional_data_sample[8])))
        directions_from_3_2 = (list(zip(directional_data_sample[9], directional_data_sample[10], directional_data_sample[11])))
        
        n = len(pts_from_3_2)
        avg_direction_from_3_2 = get_avg_direction(directions_from_3_2)
        
        # Plot stroke
        plot_markers(fig_1, pts_from_3_2, 'red')
        # Plot direction: median point + avg direction vector
        plot_line(fig_1, [pts_from_3_2[n//2]] + [tuple(a + b for a, b in zip(pts_from_3_2[n//2], avg_direction_from_3_2))], 'blue')
        # Plot head translation
        plot_markers(fig_1, [(np.sum(directional_data_sample[12])/len(directional_data_sample[12]), np.sum(directional_data_sample[13])/len(directional_data_sample[13]), np.sum(directional_data_sample[14])/len(directional_data_sample[14]))], 'black')



        # Plotting right controller
        data_sample = [] # original data of the chosen sample; 21 lists
        for col in selected_columns:
            data_sample.append(filtered_df_R[col].tolist())

        # 3.1
        #r_handed_data_sample = left_to_right_handed(data_sample=data_sample, selected_columns=selected_columns)
        # 3.2
        directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
        # 3.3
        #egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)

        pts_from_3_2 = (list(zip(directional_data_sample[0], directional_data_sample[1], directional_data_sample[2])))
        directions_from_3_2 = (list(zip(directional_data_sample[3], directional_data_sample[4], directional_data_sample[5])))
        n = len(pts_from_3_2)
        avg_direction_from_3_2 = get_avg_direction(directions_from_3_2)

        # Plot stroke
        plot_markers(fig_1, pts_from_3_2, 'orange')
        # Plot direction
        plot_line(fig_1, [pts_from_3_2[n//2]] + [tuple(a + b for a, b in zip(pts_from_3_2[n//2], avg_direction_from_3_2))], 'blue')

fig_1.show()
        

### Overview 8
Plotting left controller's egocentric coordinates without converting data from left handed to right handed.

### To Do
* Edit variables 'gesture' to any key in the dictionary and 'gesture_index' to its corresponding index

In [42]:
gesture = "PanDown"
gesture_index = 0

fig_2 = go.Figure()

for file_num, file_path in enumerate(gesture_dict[gesture][0]):
    df = pd.read_csv(file_path)
    remove_strokes(df, gesture_index, file_num)
    filtered_df_L = get_cleaned_df(df, 1, 'l')
    filtered_df_R = get_cleaned_df(df, 1, 'r')

    if (len(filtered_df_L) != 0):
        data_sample = [] # original data of the chosen sample; 21 lists
        for col in selected_columns:
            data_sample.append(filtered_df_L[col].tolist())

        # 3.1
        #r_handed_data_sample = left_to_right_handed(data_sample=data_sample, selected_columns=selected_columns)
        # 3.2
        directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
        # 3.3
        egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)

        pts_from_3_3 = (list(zip(egocentric_data_sample[6], egocentric_data_sample[7], egocentric_data_sample[8])))
        directions_from_3_3 = (list(zip(egocentric_data_sample[9], egocentric_data_sample[10], egocentric_data_sample[11])))
        
        n = len(pts_from_3_3)
        avg_direction_from_3_3 = get_avg_direction(directions_from_3_3)
        
        # Plot stroke and direction
        plot_markers(fig_2, pts_from_3_3, 'red')
        plot_line(fig_2, [pts_from_3_3[n//2]]+[tuple(x + y for x, y in zip(pts_from_3_3[n//2], avg_direction_from_3_3))], 'blue')
        
fig_2.update_layout(
    scene=dict(
        xaxis=dict(dtick=0.1),  # Adjust range and tick spacing for x-axis
        yaxis=dict(dtick=0.1),  # Adjust range and tick spacing for y-axis
        zaxis=dict(dtick=0.1),  # Adjust range and tick spacing for z-axis (if needed),
        aspectmode='data'
    )
)

fig_2.show()

### Overview #9
Functions to normalize a stroke while preserving its aspect ratio.

In [43]:
def normalize_stroke_preserving_aspect(stroke, target_range=1.0):
    # Unzip coordinates into separate x, y, and z lists
    x_vals, y_vals, z_vals = zip(*stroke)

    # Find min and max values for each axis
    min_x, max_x = min(x_vals), max(x_vals)
    min_y, max_y = min(y_vals), max(y_vals)
    min_z, max_z = min(z_vals), max(z_vals)

    # Calculate the range for each axis
    range_x = max_x - min_x
    range_y = max_y - min_y
    range_z = max_z - min_z

    # Determine the largest range to use as the scaling factor
    max_range = max(range_x, range_y, range_z)

    # Apply scaling to each point to normalize to target range
    normalized_stroke = [
        (
            ((x - min_x) / max_range) * target_range,
            ((y - min_y) / max_range) * target_range,
            ((z - min_z) / max_range) * target_range
        )
        for x, y, z in stroke
    ]

    return normalized_stroke

### Overview #10
Plotting left controller's egocentric coordinates but normalized.

In [46]:
'''PanDown 0, RotRightZ 9, ZoomOut 11'''
gesture = "PanDown"
gesture_index = 0

fig_6 = go.Figure()

for file_num, file_path in enumerate(gesture_dict[gesture][0]):
    df = pd.read_csv(file_path)
    remove_strokes(df, gesture_index, file_num)
    filtered_df_L = get_cleaned_df(df, 1, 'l')
    filtered_df_R = get_cleaned_df(df, 1, 'r')

    if (len(filtered_df_L) != 0):
        data_sample = [] # original data of the chosen sample; 21 lists
        for col in selected_columns:
            data_sample.append(filtered_df_L[col].tolist())

        # 3.1
        #r_handed_data_sample = left_to_right_handed(data_sample=data_sample, selected_columns=selected_columns)
        # 3.2
        directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
        # 3.3
        egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)

        pts_from_3_3 = (list(zip(egocentric_data_sample[6], egocentric_data_sample[7], egocentric_data_sample[8])))
        directions_from_3_3 = (list(zip(egocentric_data_sample[9], egocentric_data_sample[10], egocentric_data_sample[11])))
        
        n = len(pts_from_3_3)
        avg_direction_from_3_3 = get_avg_direction(directions_from_3_3)
        
        # Normalize stroke while preserving aspect
        norm_stroke = normalize_stroke_preserving_aspect(pts_from_3_3)
        
        # Plot stroke and direction
        plot_markers(fig_6, norm_stroke, 'red')
  
fig_6.update_layout(
    scene=dict(
        xaxis=dict(dtick=0.1),  # Adjust range and tick spacing for x-axis
        yaxis=dict(dtick=0.1),  # Adjust range and tick spacing for y-axis
        zaxis=dict(dtick=0.1),  # Adjust range and tick spacing for z-axis (if needed),
        aspectmode='data',
        camera=dict(
            eye=dict(x=1.25, y=1.25, z=0.75)  # Adjusts the viewing angle
        )
    )
)
        


fig_6.show()

### Overview #11
Functions to smooth and resample the original strokes with 50 data points.

In [12]:
def extract_n_points(coordinates):
    x, y, z = zip(*coordinates)
    # The number of control points and knots
    k = 3  # degree of the B-spline
    t = np.linspace(0, 1, len(coordinates))
    # Create the B-spline representation for each dimension
    # Condition checks that make_interp_spline will return valid values

    spl_x = make_interp_spline(t, x, k=k)
    spl_y = make_interp_spline(t, y, k=k)
    spl_z = make_interp_spline(t, z, k=k)
    # Evaluate the B-spline over a dense set of points for a smooth trajectory
    dense_t = np.linspace(0, 1, 20 * len(coordinates))  # points in the smoothed curve; can be adjusted
    x_smooth = spl_x(dense_t)
    y_smooth = spl_y(dense_t)
    z_smooth = spl_z(dense_t)
    coordinates = list(zip(x_smooth, y_smooth, z_smooth))

    # Determine the new data points based on min distance
    n_points_desired = 50
    points_list = []
    min_points_distance = round(len(x_smooth) / (n_points_desired))
    for coordinate in coordinates[::min_points_distance]:
        points_list.append(coordinate)

    if (len(points_list) < n_points_desired):
        points_list.append(coordinates[-1])
    else:
        while len(points_list) > 50:
            points_list.pop(-1)

    return points_list

### Overview #12
Plotting the average stroke after normalization and resampling 50 points.

In [ ]:
'''PanDown 0, RotRightZ 9, ZoomOut 11'''
gesture = "PanDown"
gesture_index = 0

all_strokes_L = []

fig_7 = go.Figure()

for file_num, file_path in enumerate(gesture_dict[gesture][0]):
    df = pd.read_csv(file_path)
    remove_strokes(df, gesture_index, file_num)
    filtered_df_L = get_cleaned_df(df, 1, 'l')
    filtered_df_R = get_cleaned_df(df, 1, 'r')

    if (len(filtered_df_L) != 0):
        data_sample = [] # original data of the chosen sample; 21 lists
        for col in selected_columns:
            data_sample.append(filtered_df_L[col].tolist())

        # 3.1
        #r_handed_data_sample = left_to_right_handed(data_sample=data_sample, selected_columns=selected_columns)
        # 3.2
        directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
        # 3.3
        egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)

        pts_from_3_3 = (list(zip(egocentric_data_sample[6], egocentric_data_sample[7], egocentric_data_sample[8])))
        directions_from_3_3 = (list(zip(egocentric_data_sample[9], egocentric_data_sample[10], egocentric_data_sample[11])))
        
        n = len(pts_from_3_3)
        avg_direction_from_3_3 = get_avg_direction(directions_from_3_3)
        
        # Normalize stroke while preserving aspect
        norm_stroke = normalize_stroke_preserving_aspect(pts_from_3_3)

        smoothed_stroke = extract_n_points(norm_stroke)

        all_strokes_L.append(smoothed_stroke)
        
        # Plot stroke and direction
        plot_markers(fig_7, smoothed_stroke, 'red')
        
fig_7.update_layout(
    scene=dict(
        xaxis=dict(dtick=0.1),  # Adjust range and tick spacing for x-axis
        yaxis=dict(dtick=0.1),  # Adjust range and tick spacing for y-axis
        zaxis=dict(dtick=0.1),  # Adjust range and tick spacing for z-axis (if needed),
        aspectmode='data'
    )
)     

avg_stroke_L = get_avg_stroke(all_strokes_L)

plot_markers(fig_7, avg_stroke_L, 'purple')

fig_7.show()

### Overview #13
Method 1: Output html for average stroke for left and right controller separately after normalizing and resampling stroke.

### To Do
- Edit variables 'gesture' and 'gesture_index'

In [ ]:
def output_avg_stroke_to_csv(list_of_strokes):


In [ ]:
import csv

avg_stroke_L_data_path = 'C:\\Users\\katie\\Documents\\cpsc\\VRelax\\gestureInterface\\AvgStrokeTemplates\\Method1\\avg_stroke_L_data.csv'
avg_stroke_R_data_path = 'C:\\Users\\katie\\Documents\\cpsc\\VRelax\\gestureInterface\\AvgStrokeTemplates\\Method1\\avg_stroke_R_data.csv'

header = []
list_of_avg_strokes_L = []
list_of_avg_strokes_R = []

for gesture_index, gesture in enumerate(gesture_dict.keys()):
    plot_title, file_name = gesture[0], gesture[0]
    for char in gesture[1:]:  # Start from the second character
        if char.isupper():
            plot_title += " "  # Add a space before each uppercase letter
            file_name += "_"
        plot_title += char
        file_name += char

    all_strokes_L = []
    all_strokes_R = []

    fig_L, fig_R = go.Figure(), go.Figure()
    for file_num, file_path in enumerate(gesture_dict[gesture][0]):
        
        df = pd.read_csv(file_path)
        remove_strokes(df, gesture_index, file_num)
        filtered_df_L = get_cleaned_df(df, 1, 'l')
        filtered_df_R = get_cleaned_df(df, 1, 'r')

        if (len(filtered_df_L) != 0):
            data_sample = [] # original data of the chosen sample; 21 lists
            for col in selected_columns:
                data_sample.append(filtered_df_L[col].tolist())

            # 3.1
            #r_handed_data_sample = left_to_right_handed(data_sample=data_sample, selected_columns=selected_columns)
            # 3.2
            directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
            # 3.3
            egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)

            pts_from_3_3 = (list(zip(egocentric_data_sample[6], egocentric_data_sample[7], egocentric_data_sample[8])))
            directions_from_3_3 = (list(zip(egocentric_data_sample[9], egocentric_data_sample[10], egocentric_data_sample[11])))
            
            n = len(pts_from_3_3)
            avg_direction_from_3_3 = get_avg_direction(directions_from_3_3)
            
            # Normalize stroke while preserving aspect
            norm_stroke = normalize_stroke_preserving_aspect(pts_from_3_3)

            smoothed_stroke = extract_n_points(norm_stroke)

            all_strokes_L.append(smoothed_stroke)
            
            # Plot stroke and direction
            plot_markers(fig_L, smoothed_stroke, 'pink')
            #plot_line(fig_7, [norm_stroke[n//2]]+[tuple(x + y for x, y in zip(norm_stroke[n//2], avg_direction_from_3_3))], 'blue')

        if (len(filtered_df_R) != 0):
            data_sample = [] # original data of the chosen sample; 21 lists
            for col in selected_columns:
                data_sample.append(filtered_df_R[col].tolist())
            # 3.1
            #r_handed_data_sample = left_to_right_handed(data_sample=data_sample, selected_columns=selected_columns)
            # 3.2
            directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
            # 3.3
            egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)

            pts_from_3_3 = (list(zip(egocentric_data_sample[0], egocentric_data_sample[1], egocentric_data_sample[2])))
            directions_from_3_3 = (list(zip(egocentric_data_sample[3], egocentric_data_sample[4], egocentric_data_sample[5])))
            
            n = len(pts_from_3_3)
            avg_direction_from_3_3 = get_avg_direction(directions_from_3_3)
            
            # Normalize stroke while preserving aspect
            norm_stroke = normalize_stroke_preserving_aspect(pts_from_3_3)

            smoothed_stroke = extract_n_points(norm_stroke)

            all_strokes_R.append(smoothed_stroke)
            
            # Plot stroke and direction
            plot_markers(fig_R, smoothed_stroke, 'orange')
            
    fig_L.update_layout(
        title=plot_title + " (L)",
        scene=dict(
            xaxis=dict(dtick=0.1),  # Adjust range and tick spacing for x-axis
            yaxis=dict(dtick=0.1),  # Adjust range and tick spacing for y-axis
            zaxis=dict(dtick=0.1),  # Adjust range and tick spacing for z-axis (if needed),
            aspectmode='data'
        ),
        legend=dict(
            x=.92,
            y=-.6
        ),
    )    
    fig_R.update_layout(
        title=plot_title + " (R)",
        scene=dict(
            xaxis=dict(dtick=0.1),  # Adjust range and tick spacing for x-axis
            yaxis=dict(dtick=0.1),  # Adjust range and tick spacing for y-axis
            zaxis=dict(dtick=0.1),  # Adjust range and tick spacing for z-axis (if needed),
            aspectmode='data'
        ),
        legend=dict(
            x=.92,
            y=-.6
        ),
    )   

    avg_stroke_L = get_avg_stroke(all_strokes_L)
    avg_stroke_R = get_avg_stroke(all_strokes_R)

    header.append(gesture)
    list_of_avg_strokes_L.append(avg_stroke_L)
    list_of_avg_strokes_R.append(avg_stroke_R)


    

    plot_avg(fig_L, avg_stroke_L)
    plot_avg(fig_R, avg_stroke_R)

    #fig_L.show()
    #fig_R.show()

    base_path = "C:\\Users\\katie\\Documents\\cpsc\\VRelax\\gestureInterface\\AvgStrokeTemplates\\Method1\\"
    left_path = base_path + f"{file_name}_L.html"
    right_path = base_path + f"{file_name}_R.html"
    if not os.path.exists(left_path):
        pio.write_html(fig_L, file=left_path, auto_open=False)
    if not os.path.exists(right_path):
        pio.write_html(fig_R, file=right_path, auto_open=False)

list_of_avg_strokes_L = [[f"{x},{y},{z}" for x, y, z in points] for points in zip(*list_of_avg_strokes_L)]
list_of_avg_strokes_R = [[f"{x},{y},{z}" for x, y, z in points] for points in zip(*list_of_avg_strokes_R)]

'''
with open(avg_stroke_L_data_path, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(header)  # Write the header
    writer.writerows(list_of_avg_strokes_L)
with open(avg_stroke_R_data_path, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(header)  # Write the header
    writer.writerows(list_of_avg_strokes_R)
'''

[(0.048509668965661716, 0.999770082503652, 0.27927309856549265), (0.04617169912552582, 0.9943767206780204, 0.2733597077295216), (0.04412765555339922, 0.9882823506805364, 0.26601900214898466), (0.0423181731180842, 0.9796406413918218, 0.25638298121605213), (0.039322462490915304, 0.9687153605518571, 0.2457750698860102), (0.03711174050022757, 0.9563624129625184, 0.23263361135140861), (0.035216303616611516, 0.9414087519961206, 0.22072087251993106), (0.0339721242553896, 0.9248582654106134, 0.20887777615356884), (0.03387999400469297, 0.9067032993676283, 0.1962124115500297), (0.03374878742318935, 0.8879531923158976, 0.18265301337070922), (0.0338460694219721, 0.865772797693512, 0.16834984880241122), (0.032972700929891824, 0.8415724252971906, 0.15462579545526187), (0.03217809623527483, 0.8161776744658948, 0.1422845700458207), (0.03096870565345984, 0.790065199807031, 0.1297725827563317), (0.02999176214487171, 0.7623859410008057, 0.11888971851656047), (0.029431974023100424, 0.7342293435343439, 0.1

### Method 2: Getting avg stroke after normalizing [0,1] and translating based on median point at origin

In [31]:
def med_pt_to_origin(coordinates):
    coords_array = np.array(coordinates)

    # Calculate the median coordinate
    median_coordinate = np.median(coords_array, axis=0)

    # Move everything to the origin based on the median
    translated_coordinates = coords_array - median_coordinate

    # Convert back to list of tuples if needed
    translated_coordinates_list = [tuple(coord) for coord in translated_coordinates]
    return translated_coordinates_list



In [ ]:
for gesture_index, gesture in enumerate(gesture_dict.keys()):
    plot_title = gesture[0]
    file_name = gesture[0]
    for char in gesture[1:]:  # Start from the second character
        if char.isupper():
            plot_title += " "  # Add a space before each uppercase letter
            file_name += "_"
        plot_title += char
        file_name += char

    all_strokes_L = []
    all_strokes_R = []

    fig_L, fig_R = go.Figure(), go.Figure()
    for file_num, file_path in enumerate(gesture_dict[gesture][0]):
        
        df = pd.read_csv(file_path)
        remove_strokes(df, gesture_index, file_num)
        filtered_df_L = get_cleaned_df(df, 1, 'l')
        filtered_df_R = get_cleaned_df(df, 1, 'r')

        if (len(filtered_df_L) != 0):
            data_sample = [] # original data of the chosen sample; 21 lists
            for col in selected_columns:
                data_sample.append(filtered_df_L[col].tolist())

            # 3.1
            #r_handed_data_sample = left_to_right_handed(data_sample=data_sample, selected_columns=selected_columns)
            # 3.2
            directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
            # 3.3
            egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)

            pts_from_3_3 = (list(zip(egocentric_data_sample[6], egocentric_data_sample[7], egocentric_data_sample[8])))
            directions_from_3_3 = (list(zip(egocentric_data_sample[9], egocentric_data_sample[10], egocentric_data_sample[11])))
            
            n = len(pts_from_3_3)
            avg_direction_from_3_3 = get_avg_direction(directions_from_3_3)
            
            # Normalize stroke while preserving aspect
            norm_stroke = normalize_stroke_preserving_aspect(pts_from_3_3)

            # Smooth the stroke with b-spline interpolation & extract 50 points
            smoothed_stroke = extract_n_points(norm_stroke)

            # Translate stroke based on median at origin
            translated_coordinates = med_pt_to_origin(smoothed_stroke)

            all_strokes_L.append(translated_coordinates)
            
            # Plot stroke and direction
            plot_markers(fig_L, translated_coordinates, 'pink')

        if (len(filtered_df_R) != 0):
            data_sample = [] # original data of the chosen sample; 21 lists
            for col in selected_columns:
                data_sample.append(filtered_df_R[col].tolist())
            # 3.1
            #r_handed_data_sample = left_to_right_handed(data_sample=data_sample, selected_columns=selected_columns)
            # 3.2
            directional_data_sample = quaternion_to_direction(r_handed_data_sample=data_sample, selected_columns=selected_columns)
            # 3.3
            egocentric_data_sample = world_to_headspace(directional_data_sample=directional_data_sample, directional_data_names=directional_data_names)

            pts_from_3_3 = (list(zip(egocentric_data_sample[0], egocentric_data_sample[1], egocentric_data_sample[2])))
            directions_from_3_3 = (list(zip(egocentric_data_sample[3], egocentric_data_sample[4], egocentric_data_sample[5])))
            
            n = len(pts_from_3_3)
            avg_direction_from_3_3 = get_avg_direction(directions_from_3_3)
            
            # Normalize stroke while preserving aspect
            norm_stroke = normalize_stroke_preserving_aspect(pts_from_3_3)

            # Smooth the stroke with b-spline interpolation & extract 50 points
            smoothed_stroke = extract_n_points(norm_stroke)

            # Translate stroke based on median at origin
            translated_coordinates = med_pt_to_origin(smoothed_stroke)

            all_strokes_R.append(translated_coordinates)
            
            # Plot stroke and direction
            plot_markers(fig_R, translated_coordinates, 'pink')
            
    fig_L.update_layout(
        title=plot_title + " (L)",
        scene=dict(
            xaxis=dict(dtick=0.1),  # Adjust range and tick spacing for x-axis
            yaxis=dict(dtick=0.1),  # Adjust range and tick spacing for y-axis
            zaxis=dict(dtick=0.1),  # Adjust range and tick spacing for z-axis (if needed),
            aspectmode='data'
        ),
        legend=dict(
            x=.92,
            y=-.6
        ),
    )    
    fig_R.update_layout(
        title=plot_title + " (R)",
        scene=dict(
            xaxis=dict(dtick=0.1),  # Adjust range and tick spacing for x-axis
            yaxis=dict(dtick=0.1),  # Adjust range and tick spacing for y-axis
            zaxis=dict(dtick=0.1),  # Adjust range and tick spacing for z-axis (if needed),
            aspectmode='data'
        ),
        legend=dict(
            x=.92,
            y=-.6
        ),
    )   

    avg_stroke_L = get_avg_stroke(all_strokes_L)
    avg_stroke_R = get_avg_stroke(all_strokes_R)

    plot_avg(fig_L, avg_stroke_L)
    plot_avg(fig_R, avg_stroke_R)

    #fig_L.show()
    #fig_R.show()

    base_path = "C:\\Users\\katie\\Documents\\cpsc\\VRelax\\gestureInterface\\AvgStrokeTemplates\\Method2\\"
    left_path = base_path + f"{file_name}_L.html"
    right_path = base_path + f"{file_name}_R.html"
    if not os.path.exists(left_path):
        pio.write_html(fig_L, file=left_path, auto_open=False)
    if not os.path.exists(right_path):
        pio.write_html(fig_R, file=right_path, auto_open=False)

9
50
34
50


13
50
28
50


6
50
34
50


10
50
35
50


21
50
35
50


...

21
50
36
50


22
50
32
50


17
50
35
50


29
50
33
50


32
50
34
50
